In [2]:
import sys
!{sys.executable} -m pip install scikit-learn xgboost joblib

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.22.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 8.2/8.2 MB 97.7 MB/s  0:00:00
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached narwhals-2.22.0-py3-none-any.whl (453 kB)
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   ------------------------- -------------- 23.6/36.5 MB 116.3 MB/s eta 0:00:01
   ---------------------------------------- 36.5/36.5 MB 90.8 MB/s  0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ ---------------


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\Aaditya\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load cleaned data
df = pd.read_csv('../data/processed/matches_clean.csv')

print("✅ Loaded!", df.shape)
print(df['Result'].value_counts())

✅ Loaded! (964, 15)
Result
home team win    545
away team win    240
draw             179
Name: count, dtype: int64


In [4]:
# Encode team names as numbers
le = LabelEncoder()
all_teams = pd.concat([df['Home Team Name'], df['Away Team Name']]).unique()
le.fit(all_teams)

df['Home Team Encoded'] = le.transform(df['Home Team Name'])
df['Away Team Encoded'] = le.transform(df['Away Team Name'])

# Calculate win rate for each team up to that point (no data leakage)
def get_team_stats(df):
    team_stats = {}
    home_wr, away_wr, home_avg_goals, away_avg_goals = [], [], [], []
    
    for _, row in df.iterrows():
        home = row['Home Team Name']
        away = row['Away Team Name']
        
        # Get stats before this match
        h_stats = team_stats.get(home, {'wins': 0, 'games': 0, 'goals': 0})
        a_stats = team_stats.get(away, {'wins': 0, 'games': 0, 'goals': 0})
        
        home_wr.append(h_stats['wins'] / max(h_stats['games'], 1))
        away_wr.append(a_stats['wins'] / max(a_stats['games'], 1))
        home_avg_goals.append(h_stats['goals'] / max(h_stats['games'], 1))
        away_avg_goals.append(a_stats['goals'] / max(a_stats['games'], 1))
        
        # Update stats after match
        for team, score, opp_score, won in [
            (home, row['Home Team Score'], row['Away Team Score'], row['Home Team Win']),
            (away, row['Away Team Score'], row['Home Team Score'], row['Away Team Win'])
        ]:
            if team not in team_stats:
                team_stats[team] = {'wins': 0, 'games': 0, 'goals': 0}
            team_stats[team]['games'] += 1
            team_stats[team]['wins'] += won
            team_stats[team]['goals'] += score

    df['Home Win Rate'] = home_wr
    df['Away Win Rate'] = away_wr
    df['Home Avg Goals'] = home_avg_goals
    df['Away Avg Goals'] = away_avg_goals
    return df

df = get_team_stats(df)

print("✅ Features engineered!")
print(df[['Home Team Name', 'Away Team Name', 'Home Win Rate', 'Away Win Rate']].head())

✅ Features engineered!
  Home Team Name Away Team Name  Home Win Rate  Away Win Rate
0         France         Mexico            0.0            0.0
1  United States        Belgium            0.0            0.0
2     Yugoslavia         Brazil            0.0            0.0
3        Romania           Peru            0.0            0.0
4      Argentina         France            0.0            1.0


In [5]:
# Define features and target
features = ['Home Team Encoded', 'Away Team Encoded', 
            'Home Win Rate', 'Away Win Rate',
            'Home Avg Goals', 'Away Avg Goals',
            'Extra Time', 'Penalty Shootout']

# Encode target
target_map = {'home team win': 0, 'away team win': 1, 'draw': 2}
df['Target'] = df['Result'].map(target_map)

X = df[features]
y = df['Target']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train XGBoost
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
    eval_metric='mlogloss'
)

model.fit(X_train, y_train,
          eval_set=[(X_test, y_test)],
          verbose=False)

# Evaluate
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"✅ Model trained!")
print(f"   Accuracy: {acc:.2%}")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred, 
      target_names=['Home Win', 'Away Win', 'Draw']))

✅ Model trained!
   Accuracy: 53.37%

Detailed Report:
              precision    recall  f1-score   support

    Home Win       0.61      0.80      0.69       109
    Away Win       0.36      0.33      0.35        48
        Draw       0.00      0.00      0.00        36

    accuracy                           0.53       193
   macro avg       0.33      0.38      0.35       193
weighted avg       0.44      0.53      0.48       193



In [6]:
# Add more powerful features
df['Goal Diff Avg'] = df['Home Avg Goals'] - df['Away Avg Goals']
df['Win Rate Diff'] = df['Home Win Rate'] - df['Away Win Rate']
df['Stage Encoded'] = LabelEncoder().fit_transform(df['Stage Name'].fillna('unknown'))

# Recent form — last 5 matches win rate
def recent_form(df, n=5):
    team_history = {}
    home_form, away_form = [], []
    
    for _, row in df.iterrows():
        home, away = row['Home Team Name'], row['Away Team Name']
        
        h_hist = team_history.get(home, [])
        a_hist = team_history.get(away, [])
        
        home_form.append(np.mean(h_hist[-n:]) if h_hist else 0.5)
        away_form.append(np.mean(a_hist[-n:]) if a_hist else 0.5)
        
        team_history.setdefault(home, []).append(row['Home Team Win'])
        team_history.setdefault(away, []).append(row['Away Team Win'])
    
    df['Home Recent Form'] = home_form
    df['Away Recent Form'] = away_form
    return df

df = recent_form(df)

# Updated features
features_v2 = ['Home Team Encoded', 'Away Team Encoded',
               'Home Win Rate', 'Away Win Rate',
               'Home Avg Goals', 'Away Avg Goals',
               'Goal Diff Avg', 'Win Rate Diff',
               'Home Recent Form', 'Away Recent Form',
               'Stage Encoded', 'Extra Time', 'Penalty Shootout']

X2 = df[features_v2]
y2 = df['Target']

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

model_v2 = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)

model_v2.fit(X_train2, y_train2,
             eval_set=[(X_test2, y_test2)],
             verbose=False)

y_pred2 = model_v2.predict(X_test2)
acc2 = accuracy_score(y_test2, y_pred2)
print(f"✅ Improved model trained!")
print(f"   V1 Accuracy: 53.37%")
print(f"   V2 Accuracy: {acc2:.2%}")
print(classification_report(y_test2, y_pred2,
      target_names=['Home Win', 'Away Win', 'Draw']))

✅ Improved model trained!
   V1 Accuracy: 53.37%
   V2 Accuracy: 54.40%
              precision    recall  f1-score   support

    Home Win       0.62      0.79      0.69       109
    Away Win       0.40      0.35      0.38        48
        Draw       0.17      0.06      0.08        36

    accuracy                           0.54       193
   macro avg       0.40      0.40      0.38       193
weighted avg       0.48      0.54      0.50       193



In [7]:
import os

# Save model
os.makedirs('../backend/models', exist_ok=True)

joblib.dump(model_v2, '../backend/models/match_predictor.joblib')
joblib.dump(le, '../backend/models/label_encoder.joblib')
joblib.dump(features_v2, '../backend/models/feature_names.joblib')

# Save team stats for the API to use
team_summary = df.groupby('Home Team Name').agg(
    Games=('Home Team Win', 'count'),
    Win_Rate=('Home Win Rate', 'last'),
    Avg_Goals=('Home Avg Goals', 'last'),
    Recent_Form=('Home Recent Form', 'last')
).reset_index()
team_summary.columns = ['Team', 'Games', 'Win_Rate', 'Avg_Goals', 'Recent_Form']
team_summary.to_csv('../data/processed/team_summary.csv', index=False)

print("✅ Day 2 Complete! Files saved:")
print("   backend/models/match_predictor.joblib")
print("   backend/models/label_encoder.joblib") 
print("   backend/models/feature_names.joblib")
print("   data/processed/team_summary.csv")
print(f"\nModel ready to serve predictions for {len(team_summary)} teams!")

✅ Day 2 Complete! Files saved:
   backend/models/match_predictor.joblib
   backend/models/label_encoder.joblib
   backend/models/feature_names.joblib
   data/processed/team_summary.csv

Model ready to serve predictions for 81 teams!
